# 🦙 QLoRA 微调实验室 — Colab 版

基于 `lora-lab` 项目，在 Google Colab 免费 T4 GPU（15GB）上对 **Qwen2.5-0.5B-Instruct** 进行 QLoRA 4-bit 量化微调。

## 技术要点
- **QLoRA** = Quantization（4-bit NF4 量化压缩显存）+ LoRA（低秩矩阵只训练少量参数）
- **显存对比**：全量微调 7B 需 ~100GB，QLoRA 只需 6GB，**压缩率 >90%**
- **LoRA 原理**：冻结原始权重 W，只训练低秩分解矩阵 A、B（`r=16`），前向时 `W + α/r * BA`

---
⚠️ **使用前** 确认：Runtime → Change runtime type → GPU (T4)

## Cell 1：安装依赖

安装所有必要的库。`bitsandbytes` 提供 4-bit 量化支持，`peft` 提供 LoRA 接口，`trl` 提供 SFT 训练器。

In [ ]:
# 安装微调所需全部依赖（约2-3分钟）
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl
!pip install -q sentencepiece protobuf

# 验证 GPU 可用性
import torch
print(f'✅ PyTorch 版本: {torch.__version__}')
print(f'✅ CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'✅ 显存总量: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2：挂载 Google Drive

挂载 Drive 用于保存训练好的模型。训练完成后模型会自动保存到 Drive，避免 Colab 断线后丢失。

In [ ]:
# 挂载 Google Drive，用于持久化保存模型
from google.colab import drive
drive.mount('/content/drive')

import os
# 在 Drive 中创建保存目录
OUTPUT_DIR = '/content/drive/MyDrive/lora-lab/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ 模型将保存至: {OUTPUT_DIR}')

## Cell 3：上传训练数据

上传本地的 `train.jsonl` 文件。数据格式：每行一个 JSON，包含 `instruction`（指令）和 `output`（回答）字段。

示例格式：
```json
{"instruction": "解释什么是向量数据库", "output": "向量数据库是专门用于存储和检索高维向量的数据库..."}
```

In [ ]:
# 方式一：从本地上传 train.jsonl
from google.colab import files

print('📂 请选择本地的 train.jsonl 文件...')
uploaded = files.upload()

DATA_PATH = None
for filename in uploaded.keys():
    DATA_PATH = f'/content/{filename}'
    print(f'✅ 数据文件已上传: {DATA_PATH}')

# 预览前3条数据
import json
print('\n📋 数据预览（前3条）:')
with open(DATA_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        item = json.loads(line)
        print(f'  [{i+1}] instruction: {item["instruction"][:50]}...')
        print(f'       output: {item["output"][:50]}...')

## Cell 4：配置超参数

与 `config.py` 中的 `TrainConfig` 完全对应。关键参数说明：
- `lora_r=16`：LoRA 秩，越大参数越多精度越高，显存也越大
- `lora_alpha=32`：缩放因子，通常设为 `2*r`
- `load_in_4bit=True`：QLoRA 核心，将模型量化为 4-bit NF4 格式

In [ ]:
# ========= 超参数配置（与 config.py 保持一致）=========
from dataclasses import dataclass

@dataclass
class TrainConfig:
    # 模型配置
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'  # 0.5B 参数，T4 完全可跑
    output_dir: str = OUTPUT_DIR                       # Drive 目录
    data_path: str = DATA_PATH                         # 上传的数据文件
    
    # 训练超参
    max_length: int = 512       # 序列最大长度（token 数）
    num_epochs: int = 3         # 训练轮数
    batch_size: int = 4         # 并行样本数（T4 15GB 可支持 4）
    learning_rate: float = 2e-4 # 学习率，LoRA 常用 2e-4
    
    # LoRA 配置
    lora_r: int = 16            # 低秩矩阵的秩 r（参数量 = 2 * r * d_model）
    lora_alpha: int = 32        # 缩放系数 alpha，实际缩放 = alpha/r = 2
    lora_dropout: float = 0.05  # Dropout 防过拟合
    
    # 量化配置
    load_in_4bit: bool = True   # QLoRA 核心：4-bit NF4 量化

cfg = TrainConfig()
print('✅ 超参数配置完成:')
print(f'  模型: {cfg.model_name}')
print(f'  LoRA r={cfg.lora_r}, alpha={cfg.lora_alpha}')
print(f'  4bit量化: {cfg.load_in_4bit}')
print(f'  学习率: {cfg.learning_rate}, Epochs: {cfg.num_epochs}')

## Cell 5：加载模型（BitsAndBytes 4-bit 量化）

使用 **NF4 格式**（Normal Float 4-bit）量化，是专为 QLoRA 设计的量化格式，精度损失极小。

计算精度仍使用 `float16`（不影响反向传播准确性），只有**权重存储**是 4-bit。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ===== 4-bit 量化配置（QLoRA 核心）=====
bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.load_in_4bit,
    bnb_4bit_quant_type='nf4',          # NF4 格式，专为 4-bit 优化
    bnb_4bit_compute_dtype=torch.float16, # 计算精度仍用 fp16
    bnb_4bit_use_double_quant=True,       # 双重量化，进一步节省显存
)

# ===== 加载基础模型 =====
print(f'⏳ 正在加载模型: {cfg.model_name}（约1-2分钟）...')
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=bnb_config,
    device_map='auto',    # 自动分配到 GPU
    trust_remote_code=True,
)

# ===== 加载 Tokenizer =====
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # 防止 padding 报错

# 显示显存占用
allocated = torch.cuda.memory_allocated() / 1e9
print(f'✅ 模型加载完成！当前显存占用: {allocated:.2f} GB')
print(f'✅ 模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')

## Cell 6：配置 LoRA 适配器

LoRA 只在 **注意力层的 q_proj（查询）和 v_proj（值）** 上注入低秩矩阵。

**设计要点**：为什么选 q_proj/v_proj？因为注意力机制对语言理解最关键，修改这两个权重可以以最小代价最大化提升效果。

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# ===== LoRA 配置 =====
lora_config = LoraConfig(
    r=cfg.lora_r,                          # 低秩矩阵的秩
    lora_alpha=cfg.lora_alpha,             # 缩放系数
    target_modules=['q_proj', 'v_proj'],   # 注入 LoRA 的目标层
    lora_dropout=cfg.lora_dropout,        # Dropout 防过拟合
    bias='none',                           # 不训练 bias
    task_type=TaskType.CAUSAL_LM,         # 因果语言模型
)

# 将 LoRA 适配器注入模型
model = get_peft_model(model, lora_config)

# 查看可训练参数量
model.print_trainable_parameters()
# 说明：LoRA 只训练 ~0.1% 参数，但效果接近全量微调！

print('\n✅ LoRA 适配器注入完成')
print('  可训练层: q_proj, v_proj（共 28 层 × 2 × r × d_model 参数）')

## Cell 7：开始训练

使用 `SFTTrainer`（Supervised Fine-Tuning Trainer）执行指令微调。

训练过程中会实时显示 `loss`，期望 loss 从 ~2-3 下降到 ~0.5-1.0 表示训练正常。

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

# ===== 加载并格式化数据集 =====
dataset = load_dataset('json', data_files=cfg.data_path, split='train')

def format_instruction(example):
    """将 instruction/output 格式转换为对话格式"""
    return {
        'text': f'<|im_start|>user\n{example["instruction"]}<|im_end|>\n'
                f'<|im_start|>assistant\n{example["output"]}<|im_end|>'
    }

dataset = dataset.map(format_instruction)
print(f'✅ 数据集加载完成，共 {len(dataset)} 条样本')

# ===== 训练参数配置 =====
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.batch_size,
    learning_rate=cfg.learning_rate,
    fp16=True,                  # 混合精度训练（节省显存+加速）
    logging_steps=10,           # 每10步打印一次 loss
    save_steps=100,             # 每100步保存一次 checkpoint
    save_total_limit=2,         # 最多保留2个 checkpoint
    report_to='none',           # 不上报到 wandb
    warmup_steps=10,            # 前10步线性 warmup
    lr_scheduler_type='cosine', # 余弦退火学习率调度
)

# ===== 初始化 SFT Trainer =====
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=cfg.max_length,
)

# ===== 开始训练 =====
print('🚀 开始训练...')
print(f'  总步数 ≈ {len(dataset) * cfg.num_epochs // cfg.batch_size} 步')
trainer.train()

# ===== 保存最终模型 =====
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f'\n✅ 训练完成！模型已保存至 {cfg.output_dir}')

## Cell 8：推理测试

加载训练好的 LoRA 适配器，对3条测试 prompt 进行推理，验证微调效果。

对比微调前后的输出，体现 LoRA 的效果。

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ===== 加载基础模型 + LoRA 适配器 =====
print('⏳ 加载微调后的模型...')
base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
# 将 LoRA 权重合并到基础模型（推理时更高效）
inf_model = PeftModel.from_pretrained(base_model, cfg.output_dir)
inf_model = inf_model.merge_and_unload()  # 合并 LoRA 权重，推理速度等同原始模型
inf_model.eval()

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)

def generate(prompt: str, max_new_tokens: int = 200) -> str:
    """单次推理生成"""
    # 格式化为 Qwen2.5 对话格式
    formatted = f'<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(formatted, return_tensors='pt').to(inf_model.device)
    
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # 只返回新生成的部分
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# ===== 3条测试 Prompt =====
test_prompts = [
    '请解释什么是 RAG（检索增强生成）？',
    '什么是 LoRA 微调？它和全参数微调有什么区别？',
    '向量数据库和传统关系型数据库的区别是什么？',
]

print('\n' + '='*60)
print('🧪 推理测试结果（微调后模型输出）')
print('='*60)
for i, prompt in enumerate(test_prompts, 1):
    print(f'\n【问题 {i}】{prompt}')
    answer = generate(prompt)
    print(f'【回答 {i}】{answer}')
    print('-'*40)

print('\n✅ 推理测试完成！')